# Notes from AIMA Chapter 13 Probabilistic Reasoning

# Technical Report: Probabilistic Reasoning and Bayesian Architectures

## 1. Deliverable 1: Professor-Style Technical Report

### 1.1 Introduction: Representing Knowledge in an Uncertain Domain

The transition from the logicist tradition to probabilistic reasoning reflects the need to reason under uncertainty. Logic assumes certainty, while real agents operate with partial observations and imperfect models. The full joint distribution is complete but grows as $2^n$, making direct representation infeasible. Bayesian Networks (BNs) solve this by encoding dependencies via a directed acyclic graph (DAG): nodes are random variables and edges denote direct influence.

Conditional independence is captured structurally through D-separation. In a chain ($A \to B \to C$), $A$ and $C$ become independent given $B$. In a fork ($A \leftarrow B \to C$), conditioning on $B$ blocks dependence. In a collider ($A \to B \leftarrow C$), conditioning on $B$ activates dependence (*explaining away*). These structural properties enable factorization and efficient inference.

---

### 1.2 The Semantics of Bayesian Networks

A BN defines a joint distribution through local conditional distributions:

$$
P(x_1,\dots,x_n)=\prod_{i=1}^{n} P(x_i \mid Parents(X_i))
$$

Each factor is a **local conditional distribution (CPT)** describing how a node depends only on its parents. This is an exact compression enabled by conditional independence.

Local Markov Assumption:

A node $X$ is conditionally independent of its non-descendants given $Parents(X)$.

Markov Blanket:

$$
MB(X)=Parents(X)\cup Children(X)\cup Parents(Children(X))
$$

Given its Markov blanket, a node is independent of all other variables; the blanket forms a minimal local information boundary.

Continuous variables can be represented using parametric forms (e.g., Linear Gaussian models) while preserving factorization.

---

### 1.3 Exact Inference in Bayesian Networks

Inference computes the posterior:

$$
P(X \mid e)
$$

where $X$ is a query variable and $e$ is evidence.

#### Inference by Enumeration

$$
P(X \mid e)=\alpha \sum_y P(X,e,y)
$$

$y$ are hidden variables; $\alpha$ normalizes the result so probabilities sum to 1.

#### Variable Elimination (VE)

Inference operates on factors ($\phi$). Pointwise multiplication combines compatible factors (combining constraints), and summation marginalizes variables (removing irrelevant ones). Variable ordering strongly impacts efficiency.

#### Clustering and Clique Methods

Join-tree methods group variables into cliques and propagate messages. Exact inference remains NP-hard due to potentially large clique sizes.

---

### 1.4 Approximate Inference: Stochastic Simulation

Sampling methods estimate posteriors by simulating worlds.

#### Direct Sampling

* $x_i \sim P(X_i \mid Parents(X_i))$
* Samples from the prior; ignores evidence.

#### Rejection Sampling

$$
\hat{P}(X \mid e)\approx \frac{N(X,e)}{N(e)}
$$

Samples inconsistent with evidence are discarded. Inefficient when $P(e)$ is small.

#### Likelihood Weighting (Bias Correction)

Evidence nodes are fixed rather than sampled, creating a proposal distribution $Q$. Each sample is weighted:

$$
w=\prod_i P(e_i \mid Parents(E_i))
$$

This corrects sampling bias (weights proportional to $P/Q$). Intuition: keep all samples but scale their influence according to how well they explain the evidence.

#### MCMC

Constructs a Markov chain whose stationary distribution equals the posterior.

#### Gibbs Sampling

$$
x_i \sim P(X_i \mid MB(X_i))
$$

Each update samples one non-evidence variable conditioned on its Markov blanket, producing efficient local moves that converge to the global posterior.

---

### 1.5 Causal Networks and the Logic of Intervention

Causal networks extend Bayesian networks by interpreting edges as **causal mechanisms**, not merely statistical dependencies.

#### Structural Equations

Each variable is defined by a structural equation:

$$
X_i = f_i(Parents(X_i), U_i)
$$

where:

* $f_i$ is a deterministic or stochastic mechanism,
* $U_i$ is an exogenous disturbance (unmodeled causes, noise, or error term).

These structural equations explain *how* effects arise from causes. The probabilistic CPT emerges from uncertainty in the disturbance terms $U_i$.

#### Unmodeled Variables / Disturbances

The $U_i$ variables represent external influences not explicitly included in the graph. They explain why causal models remain probabilistic even when mechanisms are deterministic. Hidden common causes can induce confounding if not modeled.

#### Causally Compatible Ordering

Variables must be ordered so that causes precede effects (a topological ordering of the DAG). This ensures edges correspond to real causal direction.

#### Intervention and the do-Operator

Observation:

$$
P(Y \mid X=x)
$$

Intervention:

$$
P(Y \mid do(X=x))
$$

Intervening replaces the structural equation for $X$ with a fixed value and removes incoming edges (“graph surgery”).

#### Back-Door Criterion (Confounding Control)

A set of variables $Z$ satisfies the back-door criterion for estimating the causal effect of $X$ on $Y$ if:

1. No member of $Z$ is a descendant of $X$.
2. $Z$ blocks all back-door paths from $X$ to $Y$ (paths entering $X$ through an incoming edge).

When this holds:

$$
P(Y \mid do(X))=\sum_z P(Y \mid X,z)P(z)
$$

Intuition: adjust for common causes so correlation approximates causal effect.

---

### 1.6 Summary and Historical Context

Bayesian Networks unified AI with statistics by introducing structured probabilistic reasoning. Causal extensions (Pearl) added intervention reasoning, enabling prediction of action outcomes rather than mere observation.

---

### 1.7 Professor’s Pedagogical Appendix

#### Common Misconceptions

* Conditional independence is structural, not absolute.
* Deterministic computation can produce probabilistic beliefs.
* Correlation is not causation without causal structure.

#### Intuition Building

* Markov blanket = local information boundary.
* Sampling approximates posteriors via simulated worlds.
* Causal models encode mechanisms via structural equations.

#### Mental Debugging: BN Pitfalls

1. Graphs must remain acyclic.
2. Poor variable order causes factor explosion.
3. Conditioning on colliders activates dependence.
4. Adjust only for valid back-door variables (avoid descendants of treatment).

#### Compact Equation Reference

| Concept              | Equation                                       | Description                   |
| -------------------- | ---------------------------------------------- | ----------------------------- |
| BN Factorization     | $P(x_1,\dots,x_n)=\prod_i P(x_i \mid Pa(X_i))$ | Joint from local conditionals |
| Bayes’ Rule          | $P(H\mid E)=\frac{P(E\mid H)P(H)}{P(E)}$       | Belief update                 |
| Enumeration          | $P(X\mid e)=\alpha \sum_y P(X,e,y)$            | Exact inference               |
| Likelihood Weight    | $w=\prod P(e_i \mid Pa(E_i))$                  | Sampling bias correction      |
| Gibbs Sampling       | $x_i \sim P(X_i \mid MB(X_i))$                 | Local MCMC update             |
| Structural Equation  | $X_i=f_i(Pa(X_i),U_i)$                         | Causal mechanism              |
| do-operator          | $P(Y \mid do(X=x))$                            | Intervention                  |
| Back-door Adjustment | $\sum_z P(Y \mid X,z)P(z)$                     | Confounding control           |

---

## 2. Deliverable 2: Ultra-Compressed Flow-Graph Quick Reference

Uncertain Domain → Bayesian Network (DAG) → Conditional Independence → Markov Blanket → Factorized Joint
$P(x_1,\dots,x_n)=\prod_i P(x_i \mid Parents(X_i))$
→ Posterior Inference $P(X \mid e)$ → Exact (Enumeration, VE) → Approximate (Sampling) → Rejection (filter) → Likelihood Weighting (reweight via $P/Q$) → MCMC → Gibbs (sample via Markov blanket) → Causal Layer → Structural Equations $X_i=f_i(Pa(X_i),U_i)$ → Disturbances $U_i$ → Intervention $do(X)$ → Back-door adjustment → Causal decision making.
